# Does the grid depend on the map projection?

Companion to `01_the_problem.md` §1.7. A region of the Earth with 4 corners has exactly one conformal
map onto a rectangle, so solving in any *conformal* projection should give the same grid on the Earth,
**if every projection sees the same region**. The catch: a straight edge between two outline vertices is
a different curve on the Earth in each projection.

This notebook solves California in all four projection kinds twice: once with the raw 30-vertex outline,
once with the edges densified along geodesics. Expected: the raw grids differ, the densified ones agree.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Geod

WORKTREE = Path("/glade/u/home/manishrv/documents/croc/regional_mom_workflows/mom6_forge-worktrees/conformal-solver")
sys.path.insert(0, str(WORKTREE))
import mom6_forge, mom6_forge._conformal as cf
assert str(WORKTREE) in mom6_forge.__file__, mom6_forge.__file__

o = json.loads((WORKTREE / "tests/data/california_bays.outline.json").read_text())["outline"]
lon, lat, corners = np.array(o["lon"]), np.array(o["lat"]), o["corners"]
geod = Geod(ellps="WGS84")
KINDS = ("lcc", "merc", "tmerc", "stere")

In [ ]:
def densify(lon, lat, corners, step_km=5.0):
    """Insert points every ~step_km along the geodesic between consecutive vertices; remap corners."""
    L, T, idx = [], [], {}
    for i in range(len(lon)):
        j = (i + 1) % len(lon)
        idx[i] = len(L)
        L.append(lon[i]); T.append(lat[i])
        k = int(geod.inv(lon[i], lat[i], lon[j], lat[j])[2] / 1e3 // step_km)
        for p in (geod.npts(lon[i], lat[i], lon[j], lat[j], k) if k else []):
            L.append(p[0]); T.append(p[1])
    return np.array(L), np.array(T), [idx[c] for c in corners]


def solve_all(lon, lat, corners, n_cells=3000, raster_points=250_000):
    return {k: cf.solve(lon, lat, corners, n_cells, k, raster_points=raster_points) for k in KINDS}


def summarise(grids, label):
    print(f"== {label}")
    ref = grids["lcc"]
    for k, cg in grids.items():
        size = cf._cell_geometry(cg)[2]
        p1, p99 = np.percentile(size, [1, 99])
        line = f"  {k:5s} nx,ny={cg.nx},{cg.ny}  aspect={cg.aspect:.4f}  size ratio p99/p1={p99/p1:.3f}"
        if k != "lcc" and cg.lon.shape == ref.lon.shape:
            d = geod.inv(ref.qlon, ref.qlat, cg.qlon, cg.qlat)[2] / 1e3
            line += f"  | nodes vs lcc: median {np.median(d)*1e3:.0f} m, max {d.max():.2f} km"
        print(line)

In [ ]:
raw = solve_all(lon, lat, corners)
summarise(raw, "raw outline (edges straight in each projection plane)")
print("  size_ratios (coarse, what 'auto' uses):", {k: round(v, 3) for k, v in cf.size_ratios(lon, lat, corners).items()})

dlon, dlat, dcorners = densify(lon, lat, corners)
dense = solve_all(dlon, dlat, dcorners)
summarise(dense, f"densified along geodesics ({len(dlon)} vertices)")
print("  size_ratios (coarse, what 'auto' uses):", {k: round(v, 3) for k, v in cf.size_ratios(dlon, dlat, dcorners).items()})

## Where do the raw-outline grids differ?

The same straight segment is a different curve in each plane. Plot the raw outline as each projection
draws it (densely sampled straight lines in that plane, mapped back to lon/lat), and the Mercator-minus-LCC
difference in modulus shows up as different nx, ny.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
for k in KINDS:
    p = cf.projection_for(k, lon, lat)
    x, y = p.to_xy(lon, lat)
    t = np.linspace(0, 1, 50)[:-1]
    xs = np.concatenate([x[i] + t * (x[(i + 1) % len(x)] - x[i]) for i in range(len(x))])
    ys = np.concatenate([y[i] + t * (y[(i + 1) % len(y)] - y[i]) for i in range(len(y))])
    ax.plot(*p.to_lonlat(xs, ys), lw=1, label=f"edges straight in {k}")
gl, gt, _ = densify(lon, lat, corners, step_km=1.0)
ax.plot(gl, gt, "k--", lw=0.8, label="geodesic edges")
ax.plot(lon[corners], lat[corners], "ro", label="corners")
ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.set_aspect(1 / np.cos(np.radians(lat.mean())))
ax.legend(fontsize=8); ax.set_title("Same vertices, different regions")
plt.show()

**Reading the result.** With geodesic edges, the aspect agrees to 4 digits and nodes agree to metres:
the projection no longer changes the answer, only where discretisation error lands. The coarse
`size_ratios` ranking then reflects noise in a 256-cell, 1,000-point solve rather than any real
difference in cell-size variation.